In [ ]:
#import sklearn
import igraph
import harmony
import anndata as ad
#import scanpy as sc

In [ ]:
import pegasus as pg
import pandas as pd
import os
from scipy import sparse, io 
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sb

In [ ]:
import matplotlib
import matplotlib.colors as col
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
cmap2 = col.LinearSegmentedColormap.from_list('own',['#A0A0A0','#000000','red','orange'])
cm.register_cmap(cmap=cmap2)

In [ ]:
pg.__version__

In [ ]:
data = pg.read_input("/cluster3/yflu/STS/scenic/loom/new/REF.h5ad")

In [ ]:
data

In [ ]:
pg.write_output(data,"/cluster3/yflu/STS/scenic/loom/new/REF.loom")

In [ ]:
data = pg.aggregate_matrices('/cluster/home/yflu/STS/pegasus/STS_sample_select_24.5.5.csv')
data

In [ ]:
data

In [ ]:
data.obs['Channel'].value_counts()

In [ ]:
cellnumber = data.obs['Channel'].value_counts()
cellnumber.to_csv("cellnumber_240505.csv")

In [ ]:
pg.qc_metrics(data,min_genes=250,max_genes=8000,mito_prefix="MT-",percent_mito=20)

In [ ]:
data

In [ ]:
df_qc = pg.get_filter_stats(data)
df_qc

In [ ]:
df_qc.to_csv("/cluster3/yflu/STS/pegasus/qc/95sample_qc_20240505.csv")

In [ ]:
pg.qcviolin(data, plot_type='gene', dpi=100, n_violin_per_panel=22, panel_size=(15,4))

In [ ]:
pg.filter_data(data)
data

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_raw_20240505.zarr.zip')

In [ ]:
data = pg.read_input("/cluster3/yflu/STS/pegasus/STS_95samples_raw_20240505.zarr.zip")

In [ ]:
no_ribo_genes =~data.var_names.str.startswith(('RPS'))
data = data[:, no_ribo_genes].copy()
no_ribo_genes =~data.var_names.str.startswith(('RPL'))
data = data[:, no_ribo_genes].copy()
no_mito_genes =~data.var_names.str.startswith(('MT-'))
data = data[:, no_mito_genes].copy()

In [ ]:
data

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_raw_20240505.zarr.zip')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_raw_20240505.zarr.zip')

In [ ]:
pg.identify_robust_genes(data)
pg.log_norm(data)

In [ ]:
help(pg.log_norm)

In [ ]:
pg.highly_variable_features(data)

In [ ]:
pg.calc_signature_score(data, 'cell_cycle_human')

In [ ]:
pg.write_output(data,'/cluster/home/yflu/STS/pegasus/STS_95samples_nomiro_raw_B_pca_20240505.zarr.zip')

In [ ]:
data = pg.read_input("/cluster/home/yflu/STS/pegasus/STS_95samples_nomiro_raw_B_pca_20240505.zarr.zip")

In [ ]:
pg.pca(data)

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_PCA_20240505.zarr.zip')

In [ ]:
data.obs['predicted_phase'].value_counts()

In [ ]:
pg.regress_out(data, attrs=['G1/S', 'G2/M'])

In [ ]:
pg.scatter(data, attrs='predicted_phase', basis='pca', dpi=130)

In [ ]:
pg.scatter(data, attrs=['predicted_phase'], basis='pca_regressed', dpi=130)

In [ ]:
pca_key = pg.run_harmony(data,rep = 'pca_regressed')

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_20240505.zarr.zip')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/pegasus/STS_91samples_nomiro_harmony_20240103.zarr.zip')

In [ ]:
pg.neighbors(data,rep="pca_regressed_harmony")
pg.louvain(data,rep="pca_regressed_harmony",class_label='louvain_labels')
pg.umap(data,rep="pca_regressed_harmony",out_basis='umap')
pg.tsne(data,rep="pca_regressed_harmony",out_basis='tsne')

In [ ]:
pg.scatter(data, ["louvain_labels"])

In [ ]:
data.obs.index

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_cluster_20240505.zarr.zip')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_cluster_20240505.zarr.zip')

In [ ]:
pg.scatter(data, attrs=['louvain_labels'], basis='umap')

In [ ]:
pg.infer_doublets(data, channel_attr = 'Channel', clust_attr = 'louvain_labels')

In [ ]:
help(pg.filter_data)

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_scrublet_20240505.zarr.zip')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_scrublet_20240505.zarr.zip')

In [ ]:
data

In [ ]:
pg.scatter(data,attrs=['doublet_score','louvain_labels'], legend_loc='on data',basis='umap')

In [ ]:
data.uns['pred_dbl_cluster'] 

In [ ]:
subdata=data[~data.obs.louvain_labels.isin(['32',"16","20","25","28"])].copy()

In [ ]:
subdata.obs.louvain_labels.cat.remove_unused_categories(inplace=True)
subdata.obs.louvain_labels.value_counts()

In [ ]:
pg.mark_doublets(data,dbl_clusts='louvain_labels:32,16,20,25,28')
data.obs['demux_type'].value_counts() 

In [ ]:
pg.scatter(data, attrs = ['demux_type'], legend_loc = ['right margin'], wspace = 0.1,alpha = [1.0, 0.8], palettes = 'demux_type:gainsboro,red')

In [ ]:
pg.qc_metrics(data,select_singlets=True,min_genes=250,max_genes=20000, percent_mito=20.0)

In [ ]:
df_qc = pg.get_filter_stats(data)
df_qc

In [ ]:
df_qc.to_csv("/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_doubletsqc_20240506.csv")

In [ ]:
pg.filter_data(data)
data

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_filter20240506.zarr.zip')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_filter20240506.zarr.zip')

In [ ]:
pg.identify_robust_genes(data)
pg.log_norm(data,base_matrix = "counts")
pg.highly_variable_features(data)
pg.calc_signature_score(data, 'cell_cycle_human')

In [ ]:
pg.pca(data)
pg.regress_out(data, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data,rep=pca_key,use_cache= False)
pg.louvain(data,rep=pca_key,class_label='louvain_labels')
pg.umap(data,rep=pca_key,out_basis='umap')
pg.tsne(data,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data,attrs=['louvain_labels'], legend_loc='on data',basis='tsne')

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_filter20250114.h5ad')

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/pegasus/STS_95samples_nomiro_harmony_filter20240506.h5ad')

In [ ]:
data_development = pg.read_input("/cluster3/yflu/STS/development/development.integrated_250205.h5ad")

In [ ]:
data_development

In [ ]:
pg.identify_robust_genes(data_development)
pg.log_norm(data_development)

In [ ]:
pg.highly_variable_features(data_development)

In [ ]:
pg.calc_signature_score(data_development, 'cell_cycle_human')

In [ ]:
pg.pca(data_development)
pg.regress_out(data_development, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_development,batch = 'orig.ident',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_development,rep=pca_key,use_cache= False)
pg.louvain(data_development,rep=pca_key,class_label='louvain_labels')
pg.umap(data_development,rep=pca_key,out_basis='umap')
pg.tsne(data_development,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_development,'/cluster3/yflu/STS/development/data_development_umap.h5ad')

In [ ]:
data.obs['Immune']=data.obs.Channel.copy()
data.obs.Immune.replace(['T779'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1311'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1250'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T143'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T827'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T196'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1250M'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T789'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T822'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1821'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1642'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T939'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T532'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1140'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T614'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T235'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T941'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T457'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1620'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T533'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T640T2'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T500'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T883'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1754'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T943'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T719'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T137'],'Immune_1',inplace=True)
data.obs.Immune.replace(['XM6'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T508'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1651'],'Immune_1',inplace=True)
data.obs.Immune.replace(['CH3'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T947'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T353'],'Immune_1',inplace=True)
data.obs.Immune.replace(['CH1'],'Immune_1',inplace=True)
data.obs.Immune.replace(['T1091'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T835'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1814'],'Immune_2',inplace=True)
data.obs.Immune.replace(['KHE001'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T552'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T503'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T640T1'],'Immune_2',inplace=True)
data.obs.Immune.replace(['KHE004'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1739'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1335'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T810'],'Immune_2',inplace=True)
data.obs.Immune.replace(['IH1'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T969'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T64'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T601'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T966'],'Immune_2',inplace=True)
data.obs.Immune.replace(['KHE006'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1254'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T392'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T559'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T840'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T599'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T963'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1100'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T1145'],'Immune_2',inplace=True)
data.obs.Immune.replace(['T854'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1815'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T98'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1699'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T997'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T729'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T356'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1314'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T611'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T442'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T391'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T937'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T887'],'Immune_3',inplace=True)
data.obs.Immune.replace(['KHE009'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1251'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1753'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T347'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T868'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1242'],'Immune_3',inplace=True)
data.obs.Immune.replace(['T1088'],'Immune_3',inplace=True)
data.obs.Immune.value_counts()

In [ ]:
data = data[data.obs.Immune.isin(["Immune_1","Immune_2","Immune_3"])].copy()

In [ ]:
data.obs.Immune.value_counts()

In [ ]:
data.obs.Immune = data.obs.Immune.cat.remove_categories(['T614L','T1091pi','T1753N','T1746','T1745','T1314N','T1254N','T1100N','T976','T969N','T943N','T924T2','T924T1','T888',"T810N","T1821N"])

In [ ]:
pg.de_analysis(data,cluster='Immune',de_key='immune_de')
marker_dict_immune = pg.markers(data,de_key='immune_de')
pg.write_results_to_excel(marker_dict_immune, "/cluster3/yflu/STS/Immune_de20240912.xlsx")

In [ ]:
data

In [ ]:
data.obs['Channel'].value_counts()

In [ ]:
cellnumber = data.obs['Channel'].value_counts()
cellnumber.to_csv("cellnumber_240722.csv")

In [ ]:
data_NF = data[data.obs.Channel.isin(["T1091","T1651","T854","T947","T969"])].copy()

In [ ]:
data_NF

In [ ]:
pg.pca(data_NF)

In [ ]:
pg.write_output(data_NF,'/cluster3/yflu/NF/NF_5samples_PCA_20240425.zarr.zip')

In [ ]:
data_NF = pg.read_input("/cluster3/yflu/NF/NF_5samples_PCA_20240425.zarr.zip")

In [ ]:
pg.regress_out(data_NF, attrs=['G1/S', 'G2/M'])

In [ ]:
pca_key = pg.run_harmony(data_NF,rep = 'pca_regressed')

In [ ]:
pg.neighbors(data_NF,rep=pca_key)
pg.louvain(data_NF,rep=pca_key,class_label='louvain_labels')
pg.umap(data_NF,rep=pca_key,out_basis='umap')
pg.tsne(data_NF,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_NF,attrs=['louvain_labels'], legend_loc='on data',basis='umap')

In [ ]:
data_NF.obs['Disease']=data_NF.obs.Channel.copy()
data_NF.obs.Disease.replace(['T1091'],'MPNST',inplace=True)
data_NF.obs.Disease.replace(['T1651'],'NF',inplace=True)
data_NF.obs.Disease.replace(['T854'],'MPNST',inplace=True)
data_NF.obs.Disease.replace(['T947'],'NF',inplace=True)
data_NF.obs.Disease.replace(['T969'],'NF',inplace=True)
data_NF.obs.Disease.value_counts()

In [ ]:
data_NF.obs.Disease.cat.remove_unused_categories(inplace=True)

In [ ]:
pg.scatter(data_NF,attrs=['Disease'], legend_loc='on data',basis='umap')

In [ ]:
pg.write_output(data_NF,'/cluster3/yflu/NF/NF_5samples_UMAP_20240425.h5ad')

In [ ]:
tumor_group = pd.read_excel("/cluster/home/yflu/STS/separated/CNVSCORE/group_all_tumor_5_6.xlsx")

In [ ]:
tumor_group

In [ ]:
data_tumor=data[data.obs.index.isin(tumor_group.barcode)].copy()

In [ ]:
data_tumor

In [ ]:
pg.identify_robust_genes(data_tumor)
pg.log_norm(data_tumor,base_matrix = "counts")
pg.highly_variable_features(data_tumor)
pg.calc_signature_score(data_tumor, 'cell_cycle_human')

In [ ]:
data_tumor

In [ ]:
data_tumor.obs['Channel'].value_counts()

In [ ]:
cellnumber_tumor = data_tumor.obs['Channel'].value_counts()
cellnumber_tumor.to_csv("cellnumber_tumor_240505.csv")

In [ ]:
normal_group = pd.read_excel("/cluster/home/yflu/STS/separated/CNVSCORE/group_all_normal_5_6.xlsx")

In [ ]:
data_normal=data[data.obs.index.isin(normal_group.barcode)].copy()

In [ ]:
data_normal

In [ ]:
data_normal.obs['Channel'].value_counts()

In [ ]:
cellnumber_normal = data_normal.obs['Channel'].value_counts()
cellnumber_normal.to_csv("cellnumber_normal_240505.csv")

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_raw_20240506.zarr.zip')
pg.write_output(data_normal,'/cluster3/yflu/STS/pegasus/STS_normal_95samples_raw_20240506.zarr.zip')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_91samples_raw_20240103.zarr.zip')
data_normal = pg.read_input('/cluster3/yflu/STS/pegasus/STS_normal_91samples_raw_20240103.zarr.zip')

In [ ]:
pg.pca(data_tumor)
pg.pca(data_normal)

In [ ]:
pg.regress_out(data_tumor, attrs=['G1/S', 'G2/M'])
pg.regress_out(data_normal, attrs=['G1/S', 'G2/M'])

In [ ]:
pca_key = pg.run_harmony(data_tumor,rep = 'pca_regressed')
pca_key = pg.run_harmony(data_normal,rep = 'pca_regressed')

In [ ]:
data_tumor

In [ ]:
data_normal

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_filter_B_umap_20240506.zarr.zip')
pg.write_output(data_normal,'/cluster3/yflu/STS/pegasus/STS_normal_95samples_nomiro_harmony_filter_B_umap_20240506.zarr.zip')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_91samples_nomiro_harmony_filter_B_umap_20240103.zarr.zip')

In [ ]:
data_normal = pg.read_input('/cluster3/yflu/STS/pegasus/STS_normal_91samples_nomiro_harmony_filter_B_umap_20240103.zarr.zip')

In [ ]:
pca_key = 'pca_regressed_harmony'

In [ ]:
pca_key

In [ ]:
pg.neighbors(data_tumor,rep=pca_key)
pg.louvain(data_tumor,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor,rep=pca_key,out_basis='tsne')

In [ ]:
pg.neighbors(data_tumor,rep=pca_key)
pg.louvain(data_tumor,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor,rep=pca_key,out_basis='tsne')

In [ ]:
pg.neighbors(data_normal,rep=pca_key)
pg.louvain(data_normal,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal,rep=pca_key,out_basis='umap')
pg.tsne(data_normal,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240506.zarr.zip')
pg.write_output(data_normal,'/cluster3/yflu/STS/pegasus/STS_normal_95samples_nomiro_harmony_nodoublet_20240506.zarr.zip')

In [ ]:
data_normal = pg.read_input('/cluster3/yflu/STS/pegasus/STS_normal_95samples_nomiro_harmony_nodoublet_20240506.zarr.zip')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240506.zarr.zip')

In [ ]:
data_tumor.obs.Channel.cat.remove_unused_categories(inplace=True)
data_tumor.obs.Channel.value_counts()

In [ ]:
samplenames = data_tumor.obs.Channel.value_counts().index.tolist()

In [ ]:
prefix = "/cluster3/yflu/STS/pegasus/loom/"
suffix = "_tumor.loom"
loompath_list = [prefix + s + suffix for s in samplenames]
loompath_list

In [ ]:
len(loompath_list)

In [ ]:
path

In [ ]:
for i in range(len(data_tumor.obs.Channel)):
    subdata_tumor = data_tumor[data_tumor.obs.Channel.isin([samplenames[i]])].copy()
    pg.write_output(subdata_tumor, loompath_list[i])
    print(samplenames[i])
    print(i)

In [ ]:
for i in range(2):
    subdata_tumor = data_tumor[data_tumor.obs.Channel.isin([samplenames[i]])].copy()
    pg.write_output(subdata_tumor, loompath_list[i])
    print(samplenames[i])
    print(i)

In [ ]:
data_tumor.obs.Channel.value_counts().index[i-1]

In [ ]:
range(1, len(data_tumor.obs.Channel.value_counts()))

In [ ]:
pg.scatter(data_tumor, ["louvain_labels"])

In [ ]:
pg.scatter(data_tumor, ["PTPRC"], groupby='louvain_labels')

In [ ]:
data_tumor=data_tumor[~data_tumor.obs.louvain_labels.isin(["12","14"])].copy()

In [ ]:
pg.pca(data_tumor)
pg.regress_out(data_tumor, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor,rep = 'pca_regressed')
pg.neighbors(data_tumor,rep=pca_key,use_cache = False)
pg.louvain(data_tumor,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240507.zarr.zip')

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240507.loom')

In [ ]:
data_tumor

In [ ]:
data_tumor = pg.read_input("/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240507.zarr.zip")

In [ ]:
data_tumor.obs.Channel.value_counts()

In [ ]:
pg.scatter(data_tumor,attrs=['Disease'], legend_loc='on data',basis='umap')

In [ ]:
data_tumor.obs['Group']=data_tumor.obs.Channel.copy()
data_tumor.obs.Group.replace(['CH1'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['CH3'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['IH1'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE001'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE004'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE006'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE009'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T1088'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1091'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1100'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1140'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1242'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1250'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1250M'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1251'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T1254'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1311'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1314'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T1335'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T137'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T143'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T1620'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1642'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1651'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1699'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1739'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1753'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1754'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1814'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1815'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T1821'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T196'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T235'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T347'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T353'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T356'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T391'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T392'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T442'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T457'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T500'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T503'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T508'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T532'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T533'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T552'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T559'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T599'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T601'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T611'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T614'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T64'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T640T1'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T640T2'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T719'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T729'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T779'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T789'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T810'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T822'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T827'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T835'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T840'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T854'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T868'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T883'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T887'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T937'],'Pecoma',inplace=True)
data_tumor.obs.Group.replace(['T939'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T941'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T943'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T947'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T963'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T966'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T969'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T98'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T997'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['XM6'],'EWS_PNET',inplace=True)
data_tumor.obs.Group = data_tumor.obs.Group.cat.remove_categories(['T1254N','T614L','T1746','T1745','T810N','T1314N','T888','T969N','T924T1','T1145','T924T2','T1100N','T1091pi','T943N','T976','T1753N',"T1821N"])
data_tumor.obs.Group.value_counts()

In [ ]:
data_tumor.obs['Disease']=data_tumor.obs.Channel.copy()
data_tumor.obs.Disease.replace(['CH1'],'Hemangioma',inplace=True)
data_tumor.obs.Disease.replace(['CH3'],'Hemangioma',inplace=True)
data_tumor.obs.Disease.replace(['IH1'],'Hemangioma',inplace=True)
data_tumor.obs.Disease.replace(['KHE001'],'KHE',inplace=True)
data_tumor.obs.Disease.replace(['KHE004'],'KHE',inplace=True)
data_tumor.obs.Disease.replace(['KHE006'],'KHE',inplace=True)
data_tumor.obs.Disease.replace(['KHE009'],'KHE',inplace=True)
data_tumor.obs.Disease.replace(['T1088'],'Schwannoma',inplace=True)
data_tumor.obs.Disease.replace(['T1091'],'MPNST',inplace=True)
data_tumor.obs.Disease.replace(['T1100'],'Undifferentiated sarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T1140'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1242'],'Schwannoma',inplace=True)
data_tumor.obs.Disease.replace(['T1250'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1250M'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1251'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T1254'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T1311'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1314'],'Angiosarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T1335'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T137'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T143'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease.replace(['T1620'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1642'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1651'],'NF',inplace=True)
data_tumor.obs.Disease.replace(['T1699'],'Aggressive fibromatosis',inplace=True)
data_tumor.obs.Disease.replace(['T1739'],'Liposarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T1753'],'Spindle cell tumor',inplace=True)
data_tumor.obs.Disease.replace(['T1754'],'Spindle cell tumor',inplace=True)
data_tumor.obs.Disease.replace(['T1814'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T1815'],'Hemangioma',inplace=True)
data_tumor.obs.Disease.replace(['T1821'],'ASPS',inplace=True)
data_tumor.obs.Disease.replace(['T196'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T235'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T347'],'Infantile fibrosarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T353'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T356'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T391'],'Undifferentiated sarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T392'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T442'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T457'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T500'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T503'],'MRT',inplace=True)
data_tumor.obs.Disease.replace(['T508'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T532'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T533'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T552'],'Synovial sarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T559'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T599'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T601'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T611'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T614'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T64'],'Infantile fibrosarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T640T1'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease.replace(['T640T2'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease.replace(['T719'],'Aggressive fibromatosis',inplace=True)
data_tumor.obs.Disease.replace(['T729'],'Infantile fibrosarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T779'],'Aggressive fibromatosis',inplace=True)
data_tumor.obs.Disease.replace(['T789'],'Aggressive fibromatosis',inplace=True)
data_tumor.obs.Disease.replace(['T810'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T822'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T827'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T835'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T840'],'IMT',inplace=True)
data_tumor.obs.Disease.replace(['T854'],'MPNST',inplace=True)
data_tumor.obs.Disease.replace(['T868'],'Hemangioma',inplace=True)
data_tumor.obs.Disease.replace(['T883'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease.replace(['T887'],'Lipoblastoma',inplace=True)
data_tumor.obs.Disease.replace(['T937'],'Pecoma',inplace=True)
data_tumor.obs.Disease.replace(['T939'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T941'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T943'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease.replace(['T947'],'NF',inplace=True)
data_tumor.obs.Disease.replace(['T963'],'RMS',inplace=True)
data_tumor.obs.Disease.replace(['T966'],'Lymphangioma',inplace=True)
data_tumor.obs.Disease.replace(['T969'],'NF',inplace=True)
data_tumor.obs.Disease.replace(['T98'],'Infantile fibrosarcoma',inplace=True)
data_tumor.obs.Disease.replace(['T997'],'Undifferentiated sarcoma',inplace=True)
data_tumor.obs.Disease.replace(['XM6'],'EWS_PNET',inplace=True)
data_tumor.obs.Disease = data_tumor.obs.Disease.cat.remove_categories(['T1254N','T614L','T1746','T1745','T810N','T1314N','T888','T969N','T924T1','T1145','T924T2','T1100N','T1091pi','T943N','T976','T1753N',"T1821N"])
data_tumor.obs.Disease.value_counts()

In [ ]:
data_tumor.obs.Disease.value_counts()

In [ ]:
pg.de_analysis(data_tumor,cluster='Group',de_key='Group_de')

In [ ]:
pg.de_analysis(data_tumor,cluster='Disease',de_key='Disease_de')

In [ ]:
marker_dict_group = pg.markers(data_tumor,de_key='Group_de')
pg.write_results_to_excel(marker_dict_group, "/cluster3/yflu/STS/louvain_labels_group_de20240912.xlsx")

In [ ]:
marker_dict_disease = pg.markers(data_tumor,de_key='Disease_de')
pg.write_results_to_excel(marker_dict_disease, "/cluster3/yflu/STS/louvain_labels_disease_de20240912.xlsx")

In [ ]:
pg.write_output(data_tumor,"/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240507.h5ad")

In [ ]:
data_tumor = pg.read_input("/cluster3/yflu/STS/pegasus/STS_tumor_95samples_nomiro_harmony_nodoublet_20240507.h5ad")

In [ ]:
pg.de_analysis(data_tumor,cluster='Channel',de_key='Channel_de')
marker_dict_group = pg.markers(data_tumor,de_key='Channel_de')
pg.write_results_to_excel(marker_dict_group, "/cluster3/yflu/STS/louvain_labels_channel_de20241230.xlsx")

In [ ]:
data_EWS = data_tumor[data_tumor.obs.Disease.isin(["EWS/PNET"])].copy()

In [ ]:
pg.identify_robust_genes(data_EWS)
pg.log_norm(data_EWS,base_matrix = "raw.X")
pg.highly_variable_features(data_EWS)
pg.calc_signature_score(data_EWS, 'cell_cycle_human')

In [ ]:
data_EWS

In [ ]:
pg.pca(data_EWS)
pg.regress_out(data_EWS, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_EWS,rep = 'pca_regressed')
pg.neighbors(data_EWS,rep=pca_key)
pg.louvain(data_EWS,rep=pca_key,class_label='louvain_labels')
pg.umap(data_EWS,rep=pca_key,out_basis='umap')
pg.tsne(data_EWS,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_EWS, ["Channel"])

In [ ]:
pg.write_output(data_EWS,"/cluster3/yflu/STS/pegasus/STS_EWS_20241226.h5ad")

In [ ]:
pg.scatter(data_tumor, ["Disease"])

In [ ]:
pg.scatter(data_tumor, ["EGFR","ERBB2"], groupby='louvain_labels')

In [ ]:
data_tumor

In [ ]:
disease = pd.DataFrame(data_tumor.obs.Disease.astype(str))    

In [ ]:
disease

In [ ]:
disease["barcode"] = disease.index

In [ ]:
disease.iloc[:,1]

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

ss = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
X = disease.iloc[:, 1]
y = disease.iloc[:, 0]
for train_index, test_index in ss.split(X, y):
    # print("TRAIN_INDEX:", train_index, "TEST_INDEX:", test_index)  # 获得索引值
    X_train, X_test = X[train_index], X[test_index]  # 训练集对应的值
    y_train, y_test = y[train_index], y[test_index]  

In [ ]:
disease["Disease"].value_counts()/len(disease["Disease"])

In [ ]:
y_test.value_counts()/len(y_test)

In [ ]:
data[data.obs.index.isin(tumor_group.barcode)].copy()

In [ ]:
data_tumor_04_test = data_tumor[data_tumor.obs.index.isin(y_test.index)].copy()

In [ ]:
pg.scatter(data_tumor_04_test,attrs=['Group'],basis='umap')

In [ ]:
pg.write_output(data_tumor_04_test,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_40sample_20240520.loom')

In [ ]:
data_tumor_04_test = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_95samples_40sample_20240520.loom')

In [ ]:
data_tumor_04_test

In [ ]:
data_tumor_04_test.obs.index

In [ ]:
pg.write_output(data_tumor_04_test,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_40sample_20240520.h5ad')

In [ ]:
data_tumor_04_test_anndata = sc.read_h5ad("/cluster3/yflu/STS/pegasus/STS_tumor_95samples_40sample_20240520.h5ad")

In [ ]:
del data_tumor_04_test_anndata.layers

In [ ]:
data_tumor_04_test_anndata.write("/cluster3/yflu/STS/pegasus/STS_tumor_95samples_40sample_20240520.h5ad")

In [ ]:
data_tumor_06_train = data_tumor[data_tumor.obs.index.isin(y_train.index)].copy()

In [ ]:
pg.write_output(data_tumor_06_train,'/cluster3/yflu/STS/pegasus/STS_tumor_95samples_60sample_20240520.loom')

In [ ]:
data_tumor.obs["Group"].value_counts

In [ ]:
data = pd.read_csv('/cluster3/yflu/STS/scenic/auc/all_auc_40.csv',index_col = 0)

In [ ]:
data

In [ ]:
transposed_data = data.T

In [ ]:
transposed_data

In [ ]:
transposed_data.to_csv('/cluster3/yflu/STS/scenic/auc/all_auc_40_transposed.csv', index=True)

In [ ]:
data_tumor_scenic = pg.read_input("/cluster3/yflu/STS/scenic/auc/all_auc_40_transposed.csv")

In [ ]:
data_tumor_scenic

In [ ]:
pg.identify_robust_genes(data_tumor_scenic)
pg.log_norm(data_tumor_scenic)

In [ ]:
pg.highly_variable_features(data_tumor_scenic)

In [ ]:
pg.pca(data_tumor_scenic)

In [ ]:
data_tumor_scenic.obs["Channel"] = data_tumor.obs["Channel"]

In [ ]:
pca_key = pg.run_harmony(data_tumor_scenic,rep = 'pca')

In [ ]:
pg.neighbors(data_tumor_scenic,rep="pca_harmony")
pg.louvain(data_tumor_scenic,rep="pca_harmony",class_label='louvain_labels_scenic')
pg.umap(data_tumor_scenic,rep="pca_harmony",out_basis='umap_scenic')
pg.tsne(data_tumor_scenic,rep="pca_harmony",out_basis='tsne_scenic')

In [ ]:
data_tumor_scenic.obs["Group"] = data_tumor.obs["Group"]

In [ ]:
pg.scatter(data_tumor_scenic,attrs=['louvain_labels_scenic'],basis='umap_scenic')

In [ ]:
pg.scatter(data_tumor_scenic,attrs=['Group'],basis='umap_scenic')

In [ ]:
pg.write_output(data_tumor_scenic,"/cluster3/yflu/STS/pegasus/data_tumor_scenic.h5ad")

In [ ]:
data_tumor_scenic = pg.read_input("/cluster3/yflu/STS/pegasus/data_tumor_scenic.h5ad")

In [ ]:
data_tumor_scenic.obs.Group.to_csv('/cluster3/yflu/STS/scenic/auc/all_auc_40_group.csv', index=True)

In [ ]:
pg.scatter(data_tumor,attrs=['Group'],basis='umap')

In [ ]:
data_tumor_scenic

In [ ]:
help(pg.read_input)

In [ ]:
data_tumor.obs.Disease

In [ ]:
pg.scatter(data_normal,attrs=['louvain_labels'], legend_loc='on data',basis='umap')

In [ ]:
pg.scatter(data_normal,attrs=['louvain_labels'], legend_loc='on data',basis='umap')

In [ ]:
data_normal

In [ ]:
data_normal.obs['Celltype']=data_normal.obs.louvain_labels.copy()
data_normal.obs.Celltype.replace(['1'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['2'],'T cells',inplace=True)
data_normal.obs.Celltype.replace(['3'],'Monocytes/macrophages',inplace=True)
data_normal.obs.Celltype.replace(['4'],'Monocytes/macrophages',inplace=True)
data_normal.obs.Celltype.replace(['5'],'Pericytes',inplace=True)
data_normal.obs.Celltype.replace(['6'],'Vascular endothelial cells',inplace=True)
data_normal.obs.Celltype.replace(['7'],'B cells',inplace=True)
data_normal.obs.Celltype.replace(['8'],'NK cells',inplace=True)
data_normal.obs.Celltype.replace(['9'],'T cells',inplace=True)
data_normal.obs.Celltype.replace(['10'],'Myocytes',inplace=True)
data_normal.obs.Celltype.replace(['11'],'Monocytes/macrophages',inplace=True)
data_normal.obs.Celltype.replace(['12'],'T cells',inplace=True)
data_normal.obs.Celltype.replace(['13'],'Proliferating cells',inplace=True)
data_normal.obs.Celltype.replace(['14'],'Monocytes/macrophages',inplace=True)
data_normal.obs.Celltype.replace(['15'],'Mast cells',inplace=True)
data_normal.obs.Celltype.replace(['16'],'Plasmacytoid dendritic cells',inplace=True)
data_normal.obs.Celltype.replace(['17'],'Schwann cells',inplace=True)
data_normal.obs.Celltype.replace(['18'],'Plasma cells',inplace=True)
data_normal.obs.Celltype.replace(['19'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['20'],'Myoblasts',inplace=True)
data_normal.obs.Celltype.replace(['21'],'Lymphatic endothelial cells',inplace=True)
data_normal.obs.Celltype.replace(['22'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['23'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['24'],'Smooth muscle cells',inplace=True)
data_normal.obs.Celltype.replace(['25'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['26'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['27'],'Smooth muscle cells',inplace=True)
data_normal.obs.Celltype.replace(['28'],'B cells',inplace=True)
data_normal.obs.Celltype.replace(['29'],'Fibroblasts',inplace=True)
data_normal.obs.Celltype.replace(['30'],'Smooth muscle cells',inplace=True)
data_normal.obs.Celltype.value_counts()

In [ ]:
data_normal.obs['Group']=data_normal.obs.Channel.copy()
data_normal.obs.Group.replace(['CH1'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['CH3'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['IH1'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['KHE001'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['KHE004'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['KHE006'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['KHE009'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T1088'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T1091'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1100'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1140'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1242'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T1250'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1250M'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1251'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T1254'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1311'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1314'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T1335'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T137'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T143'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T1620'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1642'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1651'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T1699'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1739'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1753'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1754'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1814'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1815'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T1821'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T196'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T235'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T347'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T353'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T356'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T391'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T392'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T442'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T457'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T500'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T503'],'MRT',inplace=True)
data_normal.obs.Group.replace(['T508'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T532'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T533'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T552'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T559'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T599'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T601'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T611'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T614'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T64'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T640T1'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T640T2'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T719'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T729'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T779'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T789'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T810'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T822'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T827'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T835'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T840'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T854'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T868'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T883'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T887'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T937'],'Pecoma',inplace=True)
data_normal.obs.Group.replace(['T939'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T941'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T943'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T947'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T963'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T966'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T969'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T98'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T997'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['XM6'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(["T1254N"],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T614L'],'RMS',inplace=True)
data_normal.obs.Group.replace(['T1746'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T1745'],'Muscle',inplace=True)
data_normal.obs.Group.replace(['T810N'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1314N'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T888'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T969N'],'Peripheral neural-like',inplace=True)
data_normal.obs.Group.replace(['T924T1'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T924T2'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T1100N'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1145'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T1091pi'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(['T943N'],'EWS_PNET',inplace=True)
data_normal.obs.Group.replace(['T976'],'Endothelial-like',inplace=True)
data_normal.obs.Group.replace(['T1753N'],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.replace(["T1821N"],'Mesenchymal-like',inplace=True)
data_normal.obs.Group.value_counts()

In [ ]:
pd.concat([data_normal.obs.Group,data_normal.obs.Celltype,data_normal.obs.Channel],axis=1).to_csv("data_normal_Group_celltype.csv")

In [ ]:
pg.scatter(data_normal,attrs=['Celltype'],basis='umap')

In [ ]:
pg.scatter(data_normal,attrs=['CXCL13'],basis='umap')

In [ ]:
pg.violin(data_normal, ["CXCL13"], groupby='Celltype')

In [ ]:
pg.write_output(data_normal,'/cluster3/yflu/STS/pegasus/STS_normal_95samples_nomiro_harmony_nodoublet_20240506.h5ad')

In [ ]:
data_normal = pg.read_input("/cluster3/yflu/STS/pegasus/STS_normal_95samples_nomiro_harmony_nodoublet_20240506.h5ad")

In [ ]:
data_normal_T1091pi = data_normal[data_normal.obs.Channel.isin(["T1091pi"])].copy()
pg.compo_plot(data_normal_T1091pi, 'Channel', 'Celltype')

In [ ]:
data_tumor_T1091 = data_tumor[data_tumor.obs.Channel.isin(["T1091"])].copy()

In [ ]:
pg.scatter(data_normal_T1091pi,basis='umap',attrs=['SCX',"NPW","DNAJB1","COL11A1","GADD45B","SUZ12"])

In [ ]:
pg.scatter(data_tumor_T1091,basis='umap',attrs=['SCX',"NPW","DNAJB1","COL11A1","GADD45B","SUZ12"])

In [ ]:
pg.scatter(data_normal,basis='umap',attrs=['SUZ12"])

In [ ]:
data_normal.obs.Channel.value_counts()

In [ ]:
data_normal.obs.Celltype.value_counts()

In [ ]:
data_normal_other = data_normal[data_normal.obs.Celltype.isin(["Fibroblasts","Pericytes","Vascular endothelial cells","Myocytes","Schwann cells","Myoblasts","Lymphatic endothelial cells","Smooth muscle cells"])].copy()

In [ ]:
data_normal_other

In [ ]:
pg.identify_robust_genes(data_normal_other)
pg.log_norm(data_normal_other)
pg.highly_variable_features(data_normal_other)
pg.calc_signature_score(data_normal_other, 'cell_cycle_human')
pg.pca(data_normal_other)
pg.regress_out(data_normal_other, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_other,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_other,rep=pca_key,use_cache= False)
pg.louvain(data_normal_other,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal_other,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_other,rep=pca_key,out_basis='tsne')
pg.write_output(data_normal_other,"/cluster3/yflu/STS/microenvironmnet/data_normal_other.h5ad")

In [ ]:
data_normal_fibro = data_normal[data_normal.obs.Celltype.isin(["Fibroblasts","Pericytes","Smooth muscle cells"])].copy()

In [ ]:
pg.identify_robust_genes(data_normal_fibro)
pg.log_norm(data_normal_fibro)
pg.highly_variable_features(data_normal_fibro)
pg.calc_signature_score(data_normal_fibro, 'cell_cycle_human')
pg.pca(data_normal_fibro)
pg.regress_out(data_normal_fibro, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_fibro,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_fibro,rep=pca_key,use_cache= False)
pg.louvain(data_normal_fibro,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal_fibro,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_fibro,rep=pca_key,out_basis='tsne')
pg.write_output(data_normal_fibro,"/cluster3/yflu/STS/microenvironmnet/data_normal_fibro.h5ad")

In [ ]:
pg.scatter(data_normal_fibro,attrs=["louvain_labels"], basis='umap', legend_loc='on data')

In [ ]:
pg.de_analysis(data_normal_fibro,cluster='louvain_labels',de_key='fibro_de')
marker_dict_normal_fibro = pg.markers(data_normal_fibro,de_key='fibro_de')
pg.write_results_to_excel(marker_dict_normal_fibro, "/cluster3/yflu/STS/microenvironmnet/louvain_labels_fibro_de20251125.xlsx")

In [ ]:
pg.violin(data_normal_fibro, ["TOP2A"], groupby='louvain_labels')

In [ ]:
data_normal_other = pg.read_input("/cluster3/yflu/STS/microenvironmnet/data_normal_other.h5ad")

In [ ]:
pg.scatter(data_normal_fibro,attrs=['CXCL12'], legend_loc='on data',basis='umap')

In [ ]:
pg.violin(data_normal_other, ["PTPRC"], groupby='louvain_labels')

In [ ]:
data_normal_other = data_normal_other[~data_normal_other.obs.louvain_labels.isin(["8","16","19"])].copy()

In [ ]:
pg.scatter(data_normal_other,attrs=['louvain_labels',"PTPRC"], legend_loc='on data',basis='umap')

In [ ]:
pg.identify_robust_genes(data_normal_other)
pg.log_norm(data_normal_other)
pg.highly_variable_features(data_normal_other)
pg.calc_signature_score(data_normal_other, 'cell_cycle_human')
pg.pca(data_normal_other)
pg.regress_out(data_normal_other, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_other,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_other,rep=pca_key,use_cache= False)
pg.louvain(data_normal_other,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal_other,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_other,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_normal_other,attrs=['louvain_labels',"Celltype"], legend_loc='on data',basis='umap')

In [ ]:
data_normal_other

In [ ]:
pg.write_output(data_normal_other,"/cluster3/yflu/STS/microenvironmnet/data_normal_other_250110.h5ad")

In [ ]:
data_normal_TNK = data_normal[data_normal.obs.Celltype.isin(["T cells","NK cells"])].copy()

In [ ]:
pg.identify_robust_genes(data_normal_TNK)
pg.log_norm(data_normal_TNK)
pg.highly_variable_features(data_normal_TNK)
pg.calc_signature_score(data_normal_TNK, 'cell_cycle_human')
pg.pca(data_normal_TNK)
pg.regress_out(data_normal_TNK, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_TNK,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_TNK,rep=pca_key,use_cache= False)
pg.louvain(data_normal_TNK,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal_TNK,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_TNK,rep=pca_key,out_basis='tsne')
pg.write_output(data_normal_TNK,"/cluster3/yflu/STS/microenvironmnet/data_normal_TNK.h5ad")

In [ ]:
data_normal_TNK = pg.read_input("/cluster3/yflu/STS/microenvironmnet/data_normal_TNK.h5ad")

In [ ]:
data_normal_TNK

In [ ]:
pg.neighbors(data_normal_TNK,rep="pca_regressed_harmony",use_cache= False)
pg.louvain(data_normal_TNK,rep="pca_regressed_harmony",resolution=4,class_label='louvain_labels_4')

In [ ]:
pg.scatter(data_normal_TNK,attrs=['louvain_labels_4'], legend_loc='on data',basis='umap')

In [ ]:
pg.write_output(data_normal_TNK,"/cluster3/yflu/STS/microenvironmnet/data_normal_TNK.h5ad")

In [ ]:
data_normal_M = data_normal[data_normal.obs.Celltype.isin(["Monocytes/macrophages","Mast cells","Plasmacytoid dendritic cells"])].copy()
pg.identify_robust_genes(data_normal_M)
pg.log_norm(data_normal_M)
pg.highly_variable_features(data_normal_M)
pg.calc_signature_score(data_normal_M, 'cell_cycle_human')
pg.pca(data_normal_M)
pg.regress_out(data_normal_M, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_M,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_M,rep=pca_key,use_cache= False)
pg.louvain(data_normal_M,rep=pca_key,class_label='louvain_labels')
pg.umap(data_normal_M,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_M,rep=pca_key,out_basis='tsne')
pg.write_output(data_normal_M,"/cluster3/yflu/STS/microenvironmnet/data_normal_M.h5ad")

In [ ]:
data_normal_M = pg.read_input("/cluster3/yflu/STS/microenvironmnet/data_normal_M.h5ad")

In [ ]:
data_normal_M

In [ ]:
pg.neighbors(data_normal_M,rep="pca_regressed_harmony",use_cache= False)
pg.louvain(data_normal_M,rep="pca_regressed_harmony",resolution=3,class_label='louvain_labels_3')

In [ ]:
pg.scatter(data_normal_M,attrs=['louvain_labels_3'], legend_loc='on data',basis='umap')

In [ ]:
pg.write_output(data_normal_M,"/cluster3/yflu/STS/microenvironmnet/data_normal_M.h5ad")

In [ ]:
data_normal_B = data_normal[data_normal.obs.Celltype.isin(["B cells","Plasma cells"])].copy()
pg.identify_robust_genes(data_normal_B)
pg.log_norm(data_normal_B)
pg.highly_variable_features(data_normal_B)
pg.calc_signature_score(data_normal_B, 'cell_cycle_human')
pg.pca(data_normal_B)
pg.regress_out(data_normal_B, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal_B,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_normal_B,rep=pca_key,use_cache= False)
pg.louvain(data_normal_B,rep=pca_key,class_label='louvain_labels')
pg.louvain(data_normal_B,rep="pca_regressed_harmony",resolution=3,class_label='louvain_labels_3')
pg.louvain(data_normal_B,rep="pca_regressed_harmony",resolution=4,class_label='louvain_labels_4')
pg.louvain(data_normal_B,rep="pca_regressed_harmony",resolution=5,class_label='louvain_labels_5')
pg.umap(data_normal_B,rep=pca_key,out_basis='umap')
pg.tsne(data_normal_B,rep=pca_key,out_basis='tsne')

In [ ]:
pg.louvain(data_normal_B,rep="pca_regressed_harmony",resolution=1.5,class_label='louvain_labels_1_5')

In [ ]:
pg.scatter(data_normal_B,attrs=['louvain_labels_1_5'], legend_loc='on data',basis='umap')

In [ ]:
pg.write_output(data_normal_B,"/cluster3/yflu/STS/microenvironmnet/data_normal_B.h5ad")

In [ ]:
Monocytes/macrophages           106956
T cells                          91830
Fibroblasts                      78073
Pericytes                        35865
Vascular endothelial cells       32674
B cells                          32111
NK cells                         23422
Myocytes                         18639
Proliferating cells               6632
Mast cells                        5218
Plasmacytoid dendritic cells      5160
Schwann cells                     4079
Plasma cells                      4035
Myoblasts                         1584
Lymphatic endothelial cells       1363
Smooth muscle cells                814
Name: Celltype, dtype: int64

In [ ]:
pg.louvain(data_normal,rep="pca_regressed_harmony",resolution=1.2,class_label='louvain_labels_1_2')

In [ ]:
data_normal

In [ ]:
pg.scatter(data_normal,attrs=['louvain_labels_1_2'], legend_loc='on data',basis='umap')

In [ ]:
data_normal

In [ ]:
pg.scatter(data_normal, ["ACTA2"], groupby='louvain_labels')

In [ ]:
pg.violin(data_normal, ["LYZ"], groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_normal,cluster='louvain_labels',de_key='normal_de')

In [ ]:
pg.de_analysis(data_normal,cluster='louvain_labels',de_key='normal_de')
marker_dict_normal = pg.markers(data_normal,de_key='normal_de')
pg.write_results_to_excel(marker_dict_normal, "/cluster3/yflu/STS/louvain_labels_normal_de20240521.xlsx")

In [ ]:
pip list

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
marker_dict_normal = pg.markers(data_normal,de_key='Normal_de')
pg.write_results_to_excel(marker_dict_normal, "/cluster/home/yflu/STS/pegasus/louvain_labels_normal_de20240521.xlsx")

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_no_immune_91samples_nomiro_harmony_nodoublet_20240103.h5ad')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_no_immune_91samples_nomiro_harmony_nodoublet_20240103.h5ad')

In [ ]:
cnvscore = pd.read_excel("/cluster/home/yflu/STS/pegasus_new/cnvscore/cnvscore_all.xlsx")

In [ ]:
cnvscore

In [ ]:
data_tumor_1=data_tumor[data_tumor.obs.index.isin(cnvscore.barcode)].copy()

In [ ]:
cnvscore_1 =cnvscore[cnvscore.barcode.isin(data_tumor_1.obs.index)].copy()

In [ ]:
sort_order = data_tumor_1.obs.index
cnvscore_1['barcode'] = pd.Categorical(cnvscore_1['barcode'], categories=sort_order, ordered=True)
cnvscore_1 = cnvscore_1.sort_values('barcode')

In [ ]:
data_tumor_1.obs.index[1:10]

In [ ]:
cnvscore_1.index = data_tumor_1.obs.index

In [ ]:
data_tumor_1.obs["cnvscore"] = cnvscore_1.cnvscore

In [ ]:
data_tumor_1.obs.cnvscore[1:10]

In [ ]:
pg.scatter(data_tumor_1, ["cnvscore"])

In [ ]:
pg.violin(data_tumor_1, ["cnvscore"], groupby='louvain_labels')

In [ ]:
data_tumor_1.obs['Disease']=data_tumor_1.obs.Channel.copy()
data_tumor_1.obs.Disease.replace(['CH1'],'Hemangioma',inplace=True)
data_tumor_1.obs.Disease.replace(['CH3'],'Hemangioma',inplace=True)
data_tumor_1.obs.Disease.replace(['IH1'],'Hemangioma',inplace=True)
data_tumor_1.obs.Disease.replace(['KHE001'],'KHE',inplace=True)
data_tumor_1.obs.Disease.replace(['KHE004'],'KHE',inplace=True)
data_tumor_1.obs.Disease.replace(['KHE006'],'KHE',inplace=True)
data_tumor_1.obs.Disease.replace(['KHE009'],'KHE',inplace=True)
data_tumor_1.obs.Disease.replace(['T1088'],'Schwannoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T1091'],'MPNST',inplace=True)
data_tumor_1.obs.Disease.replace(['T1100'],'Undifferentiated sarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T1140'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1242'],'Schwannoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T1250'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1250M'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1251'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T1254'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T1311'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1314'],'Angiosarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T1335'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T137'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T143'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.replace(['T1620'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1642'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T1651'],'NF',inplace=True)
data_tumor_1.obs.Disease.replace(['T1699'],'Aggressive fibromatosis',inplace=True)
data_tumor_1.obs.Disease.replace(['T1739'],'Liposarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T1753'],'Spindle cell tumor',inplace=True)
data_tumor_1.obs.Disease.replace(['T1754'],'Spindle cell tumor',inplace=True)
data_tumor_1.obs.Disease.replace(['T196'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T235'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T347'],'Infantile fibrosarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T353'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T356'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T391'],'Undifferentiated sarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T392'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T442'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T457'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T500'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T503'],'MRT',inplace=True)
data_tumor_1.obs.Disease.replace(['T508'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T532'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T533'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T552'],'Synovial sarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T559'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T599'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T601'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T611'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T614'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T64'],'Infantile fibrosarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T640T1'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.replace(['T640T2'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.replace(['T719'],'Aggressive fibromatosis',inplace=True)
data_tumor_1.obs.Disease.replace(['T729'],'Infantile fibrosarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T779'],'Aggressive fibromatosis',inplace=True)
data_tumor_1.obs.Disease.replace(['T789'],'Aggressive fibromatosis',inplace=True)
data_tumor_1.obs.Disease.replace(['T810'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T822'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T827'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T835'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T840'],'IMT',inplace=True)
data_tumor_1.obs.Disease.replace(['T854'],'MPNST',inplace=True)
data_tumor_1.obs.Disease.replace(['T868'],'Hemangioma',inplace=True)
data_tumor_1.obs.Disease.replace(['T883'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.replace(['T887'],'Lipoblastoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T937'],'Pecoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T939'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T941'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T943'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.replace(['T947'],'NF',inplace=True)
data_tumor_1.obs.Disease.replace(['T963'],'RMS',inplace=True)
data_tumor_1.obs.Disease.replace(['T966'],'Lymphangioma',inplace=True)
data_tumor_1.obs.Disease.replace(['T969'],'NF',inplace=True)
data_tumor_1.obs.Disease.replace(['T98'],'Infantile fibrosarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['T997'],'Undifferentiated sarcoma',inplace=True)
data_tumor_1.obs.Disease.replace(['XM6'],'EWS/PNET',inplace=True)
data_tumor_1.obs.Disease.value_counts()

In [ ]:
data_tumor_1.obs.Disease = data_tumor_1.obs.Disease.cat.remove_categories(['T1254N','T614L','T1746','T1745','T810N','T1314N','T888','T969N','T924T1','T1145','T924T2','T1100N','T1091pi','T943N','T976','T1753N'])
data_tumor_1.obs.Disease.value_counts()

In [ ]:
pg.violin(data_tumor_1, ["cnvscore"], groupby='Disease')

In [ ]:
pg.scatter(data_tumor_1,attrs=['Disease'],basis='umap')

In [ ]:
pg.write_output(data_tumor_1,'/cluster3/yflu/STS/pegasus/STS_tumor_1_91samples_20240104.h5ad')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_1_91samples_20240104.h5ad')

In [ ]:
data_tumor.obs['Group']=data_tumor.obs.Channel.copy()
data_tumor.obs.Group.replace(['CH1'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['CH3'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['IH1'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE001'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE004'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE006'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['KHE009'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T1088'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1091'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1100'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1140'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1242'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1250'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1250M'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1251'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T1254'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1311'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1314'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T1335'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T137'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T143'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T1620'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1642'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T1651'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T1699'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1739'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1753'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T1754'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T196'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T235'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T347'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T353'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T356'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T391'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T392'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T442'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T457'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T500'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T503'],'MRT',inplace=True)
data_tumor.obs.Group.replace(['T508'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T532'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T533'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T552'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T559'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T599'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T601'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T611'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T614'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T64'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T640T1'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T640T2'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T719'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T729'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T779'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T789'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T810'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T822'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T827'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T835'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T840'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T854'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T868'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T883'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T887'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T937'],'Pecoma',inplace=True)
data_tumor.obs.Group.replace(['T939'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T941'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T943'],'EWS_PNET',inplace=True)
data_tumor.obs.Group.replace(['T947'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T963'],'RMS',inplace=True)
data_tumor.obs.Group.replace(['T966'],'Endothelial-like',inplace=True)
data_tumor.obs.Group.replace(['T969'],'Peripheral neural-like',inplace=True)
data_tumor.obs.Group.replace(['T98'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['T997'],'Mesenchymal-like',inplace=True)
data_tumor.obs.Group.replace(['XM6'],'EWS_PNET',inplace=True)
data_tumor.obs.Group = data_tumor.obs.Group.cat.remove_categories(['T1254N','T614L','T1746','T1745','T810N','T1314N','T888','T969N','T924T1','T1145','T924T2','T1100N','T1091pi','T943N','T976','T1753N'])
data_tumor.obs.Group.value_counts()

In [ ]:
pg.write_output(data_tumor,'/cluster3/yflu/STS/pegasus/STS_tumor_91samples_20240104.h5ad')

In [ ]:
data_tumor = pg.read_input('/cluster3/yflu/STS/pegasus/STS_tumor_91samples_20240104.h5ad')

In [ ]:
data_tumor

In [ ]:
data_tumor

In [ ]:
data_tumor_scenic = pg.read_input("/cluster3/yflu/STS/scenic/auc/all_auc_40.csv")

In [ ]:
data_tumor

In [ ]:
data_tumor_scenic.obs.index

In [ ]:
data_tumor_scenic

In [ ]:
help(pg.read_input)

In [ ]:
pg.scatter(data_tumor,attrs=['Group'],basis='umap')

In [ ]:
pg.scatter(data_tumor,attrs=['Group'],basis='tsne')

In [ ]:
pg.de_analysis(data_tumor,cluster='Group',de_key='Group_de')

In [ ]:
marker_dict_Group = pg.markers(data_tumor,de_key='Group_de')

In [ ]:
pg.write_results_to_excel(marker_dict_Group, "/cluster/home/yflu/STS/pegasus_new/group_tumor_de20240305.xlsx")

In [ ]:
data_tumor_RMS = data_tumor[data_tumor.obs.Disease.isin(["RMS"])].copy()

In [ ]:
data_tumor_RMS

In [ ]:
pg.write_output(data_tumor_RMS,"/cluster3/yflu/STS/pegasus/data_tumor_RMS.h5ad")

In [ ]:
data_tumor_RMS = pg.read_input("/cluster3/yflu/STS/pegasus/data_tumor_RMS.h5ad")

In [ ]:
data_tumor_RMS

In [ ]:
pg.identify_robust_genes(data_tumor_RMS)
pg.log_norm(data_tumor_RMS)
pg.highly_variable_features(data_tumor_RMS)
pg.calc_signature_score(data_tumor_RMS, 'cell_cycle_human')

In [ ]:
pg.pca(data_tumor_RMS)

In [ ]:
pg.regress_out(data_tumor_RMS, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_RMS,batch = 'Channel',rep = 'pca_regressed')

In [ ]:
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_RMS,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_RMS,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_RMS,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_RMS,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor_RMS,"/cluster3/yflu/STS/development/data_tumor_RMS_umap.h5ad")

In [ ]:
data_tumor_RMS = pg.read_input("/cluster3/yflu/STS/development/data_tumor_RMS_umap.h5ad")

In [ ]:
data_tumor_RMS_T1620 = data_tumor_RMS[data_tumor_RMS.obs.Channel.isin(["T1620"])].copy()

In [ ]:
data_tumor_RMS_T1620

In [ ]:
pg.write_output(data_tumor_RMS_T1620,"/cluster3/yflu/STS/separated/T1620/data_tumor_RMS_T1620_umap.h5ad")

In [ ]:
pg.scatter(data_tumor_RMS_T1620,attrs=['Disease'],basis='umap')

In [ ]:
pg.scatter(data_tumor_RMS,attrs=['Disease'],basis='umap')

In [ ]:
pg.scatter(data_tumor_RMS,attrs=['louvain_labels'],basis='umap')

In [ ]:
pg.de_analysis(data_tumor_RMS,cluster='louvain_labels',de_key='louvain_labels_de')

In [ ]:
marker_dict_RMS = pg.markers(data_tumor_RMS,de_key='louvain_labels_de')

In [ ]:
pg.write_results_to_excel(marker_dict_RMS, "/cluster/home/yflu/STS/pegasus_new/RMS/louvain_labels_RMS_de20240130.xlsx")

In [ ]:
data_tumor_MESEN = data_tumor[data_tumor.obs.Disease.isin(["Aggressive fibromatosis","MPNST","Infantile fibrosarcoma","IMT",
                                                          "Spindle cell tumor","Undifferentiated sarcoma","Synovial sarcoma",
                                                          "Lipoblastoma","Liposarcoma"])].copy()

In [ ]:
pg.identify_robust_genes(data_tumor_MESEN)
pg.log_norm(data_tumor_MESEN)
pg.highly_variable_features(data_tumor_MESEN)
pg.calc_signature_score(data_tumor_MESEN, 'cell_cycle_human')

In [ ]:
pg.pca(data_tumor_MESEN)
pg.regress_out(data_tumor_MESEN, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_MESEN,batch = 'Channel',rep = 'pca_regressed')

In [ ]:
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_MESEN,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_MESEN,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_MESEN,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_MESEN,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_tumor_MESEN,attrs=['louvain_labels'],basis='umap',legend_loc = 'on data')

In [ ]:
pg.scatter(data_tumor_MESEN,attrs=['Disease'],basis='umap')
pg.scatter(data_tumor_MESEN,attrs=['Disease'],basis='umap',legend_loc = 'on data')

In [ ]:
pg.scatter(data_tumor_ENDO,attrs=['Disease'],basis='umap',legend_loc="on data")

In [ ]:
pg.scatter(data_tumor_NEURO,attrs=['Disease'],basis='umap')

In [ ]:
pg.scatter(data_tumor_MRT,attrs=['Cha'],basis='umap',legend_loc="on data")

In [ ]:
pg.write_output(data_tumor_MESEN,"/cluster3/yflu/STS/data_tumor_MESEN.h5ad")

In [ ]:
data_tumor_MESEN = pg.read_input("/cluster3/yflu/STS/data_tumor_MESEN.h5ad")

In [ ]:
pg.scatter(data_tumor_MESEN, ['CXCL14','IGFBP2','RORB','NR2F1','WISP2',"FGF7","POU3F1"])

In [ ]:
pg.violin(data_tumor_MESEN, ['CXCL14','IGFBP2','RORB','NR2F1','PAX7'],groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_tumor_MESEN,cluster='louvain_labels',de_key='louvain_labels_de')
marker_dict_MESEN = pg.markers(data_tumor_MESEN,de_key='louvain_labels_de')
pg.write_results_to_excel(marker_dict_MESEN, "/cluster3/yflu/STS/louvain_labels_MESEN_de20240227.xlsx")

In [ ]:
data_tumor_ENDO = data_tumor[data_tumor.obs.Disease.isin(["Hemangioma","Angiosarcoma","Lymphangioma","KHE"])].copy()
pg.identify_robust_genes(data_tumor_ENDO)
pg.log_norm(data_tumor_ENDO)
pg.highly_variable_features(data_tumor_ENDO)
pg.calc_signature_score(data_tumor_ENDO, 'cell_cycle_human')
pg.pca(data_tumor_ENDO)
pg.regress_out(data_tumor_ENDO, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_ENDO,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_ENDO,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_ENDO,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_ENDO,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_ENDO,rep=pca_key,out_basis='tsne')

In [ ]:
RMS                         106423
Aggressive fibromatosis      30412
EWS/PNET                     30333
MRT                          26209
MPNST                        13625
Infantile fibrosarcoma        8252
IMT                           5915
Spindle cell tumor            5575
Undifferentiated sarcoma      5208
KHE                           4779
Synovial sarcoma              2407
NF                            2142
Hemangioma                    2066
Lipoblastoma                  1224
Pecoma                        1072
Angiosarcoma                  1025
Liposarcoma                   1002
Lymphangioma                   545
Schwannoma                     266


In [ ]:
pg.scatter(data_tumor_ENDO,attrs=['Disease'],basis='umap',legend_loc="on data")

In [ ]:
pg.write_output(data_tumor_ENDO,"/cluster3/yflu/STS/data_tumor_ENDO.h5ad")

In [ ]:
data_tumor_ENDO = pg.read_input("/cluster3/yflu/STS/data_tumor_ENDO.h5ad")

In [ ]:
pg.scatter(data_tumor_ENDO, ['IGF2_ENSG00000167244'])
pg.violin(data_tumor_ENDO, ['SPARCL1','RNASE1','IFI27','CLEC14A','VWF','ITM2A','CD74','PALMD','ACKR1','ENTPD1'],groupby='louvain_labels')

In [ ]:
data_tumor_ENDO.obs.Channel.value_counts()

In [ ]:
pg.de_analysis(data_tumor_ENDO,cluster='louvain_labels',de_key='louvain_labels_de')
marker_dict_ENDO = pg.markers(data_tumor_ENDO,de_key='louvain_labels_de')
pg.write_results_to_excel(marker_dict_ENDO, "/cluster3/yflu/STS/louvain_labels_ENDO_de20240227.xlsx")

In [ ]:
data_tumor_NEURO = data_tumor[data_tumor.obs.Disease.isin(["NF","Schwannoma"])].copy()
pg.identify_robust_genes(data_tumor_NEURO)
pg.log_norm(data_tumor_NEURO)
pg.highly_variable_features(data_tumor_NEURO)
pg.calc_signature_score(data_tumor_NEURO, 'cell_cycle_human')
pg.pca(data_tumor_NEURO)
pg.regress_out(data_tumor_NEURO, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_NEURO,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_NEURO,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_NEURO,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_NEURO,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_NEURO,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_tumor_NEURO,attrs=['Disease'],basis='umap')

In [ ]:
pg.write_output(data_tumor_NEURO,"/cluster3/yflu/STS/data_tumor_NEURO.h5ad")

In [ ]:
data_tumor_NEURO = pg.read_input("/cluster3/yflu/STS/data_tumor_NEURO.h5ad")

In [ ]:
pg.scatter(data_tumor_NEURO, ['VEGFA','WFDC2','LOX','MT3','ZFAS1','TGM2','XIST','SAT1','OLFM1','IGFBP6'])
pg.violin(data_tumor_NEURO, ['VEGFA','WFDC2','LOX','MT3','ZFAS1','TGM2','XIST','SAT1','OLFM1','IGFBP6'],groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_tumor_NEURO,cluster='louvain_labels',de_key='louvain_labels_de')
marker_dict_NEURO = pg.markers(data_tumor_NEURO,de_key='louvain_labels_de')
pg.write_results_to_excel(marker_dict_NEURO, "/cluster3/yflu/STS/louvain_labels_NEURO_de20240227.xlsx")

In [ ]:
data_tumor_MRT = data_tumor[data_tumor.obs.Disease.isin(["MRT"])].copy()
pg.identify_robust_genes(data_tumor_MRT)
pg.log_norm(data_tumor_MRT)
pg.highly_variable_features(data_tumor_MRT)
pg.calc_signature_score(data_tumor_MRT, 'cell_cycle_human')
pg.pca(data_tumor_MRT)
pg.regress_out(data_tumor_MRT, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_MRT,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_MRT,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_MRT,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_MRT,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_MRT,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_tumor_MRT,attrs=['louvain_labels'],basis='umap',legend_loc="on data")

In [ ]:
pg.write_output(data_tumor_MRT,"/cluster3/yflu/STS/data_tumor_MRT.h5ad")

In [ ]:
data_tumor_MRT = pg.read_input("/cluster3/yflu/STS/data_tumor_MRT.h5ad")

In [ ]:
pg.scatter(data_tumor_MRT, ['IFI27','GNG11','IGFBP7','ISG15','VIM','RAMP2','CAV1','VAMP5','B2M','ANXA2'])
pg.violin(data_tumor_MRT, ['IFI27','GNG11','IGFBP7','ISG15','VIM','RAMP2','CAV1','VAMP5','B2M','ANXA2'],groupby='louvain_labels')

In [ ]:
pg.violin(data_tumor_MRT, ['IFI27','GNG11','IGFBP7','ISG15','VIM','RAMP2','CAV1','VAMP5','B2M','ANXA2'],groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_tumor_MRT,cluster='louvain_labels',de_key='louvain_labels_de')
marker_dict_MRT = pg.markers(data_tumor_MRT,de_key='louvain_labels_de')
pg.write_results_to_excel(marker_dict_MRT, "/cluster3/yflu/STS/louvain_labels_MRT_de20240227.xlsx")

In [ ]:
data_tumor_PECO = data_tumor[data_tumor.obs.Disease.isin(["Pecoma"])].copy()
pg.identify_robust_genes(data_tumor_PECO)
pg.log_norm(data_tumor_PECO)
pg.highly_variable_features(data_tumor_PECO)
pg.calc_signature_score(data_tumor_PECO, 'cell_cycle_human')
pg.pca(data_tumor_PECO)
pg.regress_out(data_tumor_PECO, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_PECO,batch = 'Channel',rep = 'pca_regressed')
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_PECO,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_PECO,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_PECO,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_PECO,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_tumor_PECO,attrs=['louvain_labels'],basis='umap',legend_loc="on data")

In [ ]:
pg.write_output(data_tumor_PECO,"/cluster3/yflu/STS/data_tumor_PECO.h5ad")

In [ ]:
data_tumor_PECO = pg.read_input("/cluster3/yflu/STS/data_tumor_PECO.h5ad")

In [ ]:
pg.scatter(data_tumor_PECO, ['PVALB','EPHX1','MGST2','TMEM101','PNKD','ZDHHC19','ENO3','ATP5MC1','CPVL','CST2'])
pg.violin(data_tumor_PECO, ['PVALB','EPHX1','MGST2','TMEM101','PNKD','ZDHHC19','ENO3','ATP5MC1','CPVL','CST2'],groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_tumor_PECO,cluster='louvain_labels',de_key='louvain_labels_de')
marker_dict_PECO = pg.markers(data_tumor_PECO,de_key='louvain_labels_de')
pg.write_results_to_excel(marker_dict_PECO, "/cluster3/yflu/STS/louvain_labels_PECO_de20240227.xlsx")

In [ ]:
data_tumor_EWS = data_tumor[data_tumor.obs.Disease.isin(["EWS/PNET"])].copy()

In [ ]:
pg.identify_robust_genes(data_tumor_EWS)
pg.log_norm(data_tumor_EWS)
pg.highly_variable_features(data_tumor_EWS)
pg.calc_signature_score(data_tumor_EWS, 'cell_cycle_human')

In [ ]:
pg.pca(data_tumor_EWS)

In [ ]:
pg.regress_out(data_tumor_EWS, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor_EWS,batch = 'Channel',rep = 'pca_regressed')

In [ ]:
pca_key = "pca_regressed_harmony"
pg.neighbors(data_tumor_EWS,rep=pca_key,use_cache= False)
pg.louvain(data_tumor_EWS,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor_EWS,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor_EWS,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor_EWS,"/cluster3/yflu/STS/EWS/data_tumor_EWS.h5ad")

In [ ]:
data_tumor_EWS = pg.read_input("/cluster3/yflu/STS/EWS/data_tumor_EWS.h5ad")

In [ ]:
pg.scatter(data_tumor_EWS,attrs=['louvain_labels'],basis='umap',legend_loc='on data')

In [ ]:
pg.scatter(data_tumor_EWS, ['TSPAN13'])

In [ ]:
pg.violin(data_tumor_EWS, ['TSPAN13'], groupby='louvain_labels')

In [ ]:
pg.de_analysis(data_tumor_EWS,cluster='louvain_labels',de_key='louvain_labels_de')

In [ ]:
marker_dict_EWS = pg.markers(data_tumor_EWS,de_key='louvain_labels_de')

In [ ]:
pg.write_results_to_excel(marker_dict_EWS, "/cluster3/yflu/STS/EWS/louvain_labels_EWS_de20240222.xlsx")

In [ ]:
data_RMS_test = pg.read_input("/cluster3/yflu/STS/RMS/STS.pega.RMS.velo.test.h5ad")

In [ ]:
data_RMS_test

In [ ]:
pg.neighbors(data_RMS_test,rep="pca_regressed_harmony")
pg.diffmap(data_RMS_test, rep="pca_regressed_harmony")

In [ ]:
pg.fle(data_RMS_test)

In [ ]:
pg.scatter(data_RMS_test, attrs='celltype', basis='fle')

In [ ]:
pg.write_output(data_RMS_test,"/cluster3/yflu/STS/RMS/STS.pega.RMS.fle.h5ad")

In [ ]:
T1620 = pg.read_input("/cluster3/yflu/STS/RMS/STS.pega.RMS.velo.T1620.h5ad")

In [ ]:
T1620

In [ ]:
pg.neighbors(T1620,rep="pca_regressed_harmony")
pg.diffmap(T1620, rep="pca_regressed_harmony")
pg.fle(T1620)

In [ ]:
pg.scatter(T1620, attrs='celltype', basis='fle')

In [ ]:
pg.umap(T1620,rep="pca_regressed_harmony",out_basis='umap1')

In [ ]:
pg.scatter(T1620, attrs='celltype', basis='umap1')

In [ ]:
T1620

In [ ]:
pg.write_output(T1620,"/cluster3/yflu/STS/RMS/T1620.fle.h5ad")

In [ ]:
T1642 = pg.read_input("/cluster3/yflu/STS/RMS/STS.pega.RMS.velo.T1642.h5ad")

In [ ]:
pg.neighbors(T1642,rep="pca_regressed_harmony")
pg.diffmap(T1642, rep="pca_regressed_harmony")
pg.fle(T1642)

In [ ]:
pg.scatter(T1642, attrs='celltype', basis='fle')

In [ ]:
pg.umap(T1642,rep="pca_regressed_harmony",out_basis='umap1')

In [ ]:
pg.scatter(T1642, attrs='celltype', basis='umap1')

In [ ]:
pg.write_output(T1642,"/cluster3/yflu/STS/RMS/T1642.fle.h5ad")

In [ ]:
data = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88samples_nomiro_harmony_nodoublet_20231104.zarr.zip')

In [ ]:
data = pg.read_input('STS_88samples_nomiro_filter_de_copykat_afteranno20231104.zarr')

In [ ]:
pg.compo_plot(data, 'Disease', 'Celltype')

In [ ]:
pg.compo_plot(data, 'Disease', 'copykat')

In [ ]:
data

In [ ]:
data.obs.Channel_1 = [bc[17:len(bc)] for bc in data.obs.index]
data.obs["Channel_1"] = data.obs.Channel_1
data.obs["Channel_1"]

In [ ]:
data

In [ ]:
data.obs['Tissue']=data.obs.Channel.copy()
data.obs.Tissue.replace(['CH1'],'Tumor',inplace=True)
data.obs.Tissue.replace(['CH3'],'Tumor',inplace=True)
data.obs.Tissue.replace(['IH1'],'Tumor',inplace=True)
data.obs.Tissue.replace(['KHE001'],'Tumor',inplace=True)
data.obs.Tissue.replace(['KHE004'],'Tumor',inplace=True)
data.obs.Tissue.replace(['KHE006'],'Tumor',inplace=True)
data.obs.Tissue.replace(['KHE009'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1088'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1091'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1091pi'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T1100'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1100N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T1140'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1145'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1242'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1250'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1250M'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1251'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1254'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1254N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T1311'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1314'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1314N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T1335'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T137'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T143'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1620'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1642'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1651'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T1699'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T196'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T235'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T347'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T353'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T356'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T391'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T392'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T442'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T457'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T487'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T500'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T503'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T508'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T532'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T533'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T552'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T559'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T599'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T601'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T610'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T611'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T614'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T614L'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T640T1'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T640T2'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T64'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T719'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T729'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T779'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T789'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T810'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T810N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T822'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T827'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T835'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T840'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T854'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T868'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T883'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T887'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T888'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T924T1'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T924T2'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T937'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T939'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T941'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T943'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T943N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T947C'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T963'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T966'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T969'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T969N'],'Peritumor',inplace=True)
data.obs.Tissue.replace(['T976'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T98'],'Tumor',inplace=True)
data.obs.Tissue.replace(['T997'],'Tumor',inplace=True)
data.obs.Tissue.replace(['XM6'],'Tumor',inplace=True)
data.obs.Tissue.value_counts()

In [ ]:
data.obs['Tissue_1']=data.obs.Channel.copy()
data.obs.Tissue_1.replace(['CH1'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['CH3'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['IH1'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['KHE001'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['KHE004'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['KHE006'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['KHE009'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1088'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1091'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1091pi'],'Skin',inplace=True)
data.obs.Tissue_1.replace(['T1100'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1100N'],'Liver',inplace=True)
data.obs.Tissue_1.replace(['T1140'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1145'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1242'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1250'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1250M'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1251'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1254'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1254N'],'Muscle',inplace=True)
data.obs.Tissue_1.replace(['T1311'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1314'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1314N'],'Liver',inplace=True)
data.obs.Tissue_1.replace(['T1335'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T137'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T143'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1620'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1642'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1651'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T1699'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T196'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T235'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T347'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T353'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T356'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T391'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T392'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T442'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T457'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T487'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T500'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T503'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T508'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T532'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T533'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T552'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T559'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T599'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T601'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T610'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T611'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T614'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T614L'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T640T1'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T640T2'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T64'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T719'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T729'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T779'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T789'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T810'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T810N'],'Lung',inplace=True)
data.obs.Tissue_1.replace(['T822'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T827'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T835'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T840'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T854'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T868'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T883'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T887'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T888'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T924T1'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T924T2'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T937'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T939'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T941'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T943'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T943N'],'Muscle',inplace=True)
data.obs.Tissue_1.replace(['T947C'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T963'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T966'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T969'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T969N'],'Stomach',inplace=True)
data.obs.Tissue_1.replace(['T976'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T98'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['T997'],'Tumor',inplace=True)
data.obs.Tissue_1.replace(['XM6'],'Tumor',inplace=True)
data.obs.Tissue_1.value_counts()

In [ ]:
data.obs['Disease']=data.obs.Channel.copy()
data.obs.Disease.replace(['CH1'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['CH3'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['IH1'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['KHE001'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE004'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE006'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE009'],'KHE',inplace=True)
data.obs.Disease.replace(['T1088'],'Schwannoma',inplace=True)
data.obs.Disease.replace(['T1091'],'MPNST',inplace=True)
data.obs.Disease.replace(['T1091pi'],'MPNST - skin',inplace=True)
data.obs.Disease.replace(['T1100'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['T1100N'],'Undifferentiated sarcoma - liver',inplace=True)
data.obs.Disease.replace(['T1140'],'RMS',inplace=True)
data.obs.Disease.replace(['T1145'],'IMT',inplace=True)
data.obs.Disease.replace(['T1242'],'Schwannoma',inplace=True)
data.obs.Disease.replace(['T1250'],'RMS',inplace=True)
data.obs.Disease.replace(['T1250M'],'RMS',inplace=True)
data.obs.Disease.replace(['T1251'],'MRT',inplace=True)
data.obs.Disease.replace(['T1254'],'IMT',inplace=True)
data.obs.Disease.replace(['T1254N'],'IMT - muscle',inplace=True)
data.obs.Disease.replace(['T1311'],'RMS',inplace=True)
data.obs.Disease.replace(['T1314'],'Angiosarcoma',inplace=True)
data.obs.Disease.replace(['T1314N'],'Angiosarcoma - liver',inplace=True)
data.obs.Disease.replace(['T1335'],'MRT',inplace=True)
data.obs.Disease.replace(['T137'],'RMS',inplace=True)
data.obs.Disease.replace(['T143'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T1620'],'RMS',inplace=True)
data.obs.Disease.replace(['T1642'],'RMS',inplace=True)
data.obs.Disease.replace(['T1651'],'NF/perineurioma',inplace=True)
data.obs.Disease.replace(['T1699'],'STS',inplace=True)
data.obs.Disease.replace(['T196'],'MRT',inplace=True)
data.obs.Disease.replace(['T235'],'RMS',inplace=True)
data.obs.Disease.replace(['T347'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T353'],'RMS',inplace=True)
data.obs.Disease.replace(['T356'],'RMS',inplace=True)
data.obs.Disease.replace(['T391'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['T392'],'RMS',inplace=True)
data.obs.Disease.replace(['T442'],'MRT',inplace=True)
data.obs.Disease.replace(['T457'],'MRT',inplace=True)
data.obs.Disease.replace(['T487'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T500'],'MRT',inplace=True)
data.obs.Disease.replace(['T503'],'MRT',inplace=True)
data.obs.Disease.replace(['T508'],'IMT',inplace=True)
data.obs.Disease.replace(['T532'],'RMS',inplace=True)
data.obs.Disease.replace(['T533'],'RMS',inplace=True)
data.obs.Disease.replace(['T552'],'Synovial sarcoma',inplace=True)
data.obs.Disease.replace(['T559'],'RMS',inplace=True)
data.obs.Disease.replace(['T599'],'IMT',inplace=True)
data.obs.Disease.replace(['T601'],'RMS',inplace=True)
data.obs.Disease.replace(['T610'],'Juvenile xanthogranuloma',inplace=True)
data.obs.Disease.replace(['T611'],'Myofibroblastic tumor',inplace=True)
data.obs.Disease.replace(['T614'],'RMS',inplace=True)
data.obs.Disease.replace(['T614L'],'RMS - lymph node',inplace=True)
data.obs.Disease.replace(['T640T1'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T640T2'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T64'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T719'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T729'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T779'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T789'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T810'],'IMT',inplace=True)
data.obs.Disease.replace(['T810N'],'IMT - lung',inplace=True)
data.obs.Disease.replace(['T822'],'RMS',inplace=True)
data.obs.Disease.replace(['T827'],'RMS',inplace=True)
data.obs.Disease.replace(['T835'],'RMS',inplace=True)
data.obs.Disease.replace(['T840'],'IMT',inplace=True)
data.obs.Disease.replace(['T854'],'MPNST',inplace=True)
data.obs.Disease.replace(['T868'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['T883'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T887'],'Lipoblastoma',inplace=True)
data.obs.Disease.replace(['T888'],'Spindle cell tumor',inplace=True)
data.obs.Disease.replace(['T924T1'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T924T2'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T937'],'Pecoma',inplace=True)
data.obs.Disease.replace(['T939'],'RMS',inplace=True)
data.obs.Disease.replace(['T941'],'RMS',inplace=True)
data.obs.Disease.replace(['T943'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T943N'],'EWS/PNET - muscle',inplace=True)
data.obs.Disease.replace(['T947C'],'NF',inplace=True)
data.obs.Disease.replace(['T963'],'RMS',inplace=True)
data.obs.Disease.replace(['T966'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T969'],'NF',inplace=True)
data.obs.Disease.replace(['T969N'],'NF - stomach',inplace=True)
data.obs.Disease.replace(['T976'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T98'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T997'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['XM6'],'EWS/PNET',inplace=True)
data.obs.Disease.value_counts()

In [ ]:
pg.scatter(data,attrs=["Tissue","Tissue_1",'Disease'],basis='umap')

In [ ]:
pg.write_output(data,'/cluster/home/yflu/STS/pegasus/STS_88samples_meta_20231104.zarr')

In [ ]:
pg.scatter(data, attrs=["louvain_labels"], basis=['umap',"tsne"],legend_loc='on data')

In [ ]:
pg.scatter(data, ["PECAM1","EPCAM","PTPRC","COL3A1"], groupby='louvain_labels')

In [ ]:
pg.scatter(data, ["MYOD1","MYOG"], groupby='louvain_labels')

In [ ]:
pg.scatter(data, attrs=["Disease"], basis='tsne')

In [ ]:
pg.de_analysis(data,cluster='louvain_labels',de_key='louvain_labels_de')

In [ ]:
marker_dict = pg.markers(data,de_key='louvain_labels_de')

In [ ]:
pg.write_results_to_excel(marker_dict, "/cluster/home/yflu/STS/pegasus/louvain_labels_de20231104.xlsx")

In [ ]:
pg.write_output(data,'/cluster/home/yflu/STS/pegasus/STS_88samples_meta_de_20231104.zarr')

In [ ]:
data = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88samples_meta_de_20231104.zarr')

In [ ]:
genes = ["SMARCB1", "MYLPF","MYL1","MYOD1","COL3A1","CD34", "ACTA2", "DLK1","S100B","TAGLN","COL1A1","ENG","VWF","PTPRC","EPCAM","CD3D","CD8A","CD4","NKG7","GNLY","CD14","FCGR3A","AIF1","CD68","LILRA4","CLEC4C","LYZ"]

In [ ]:
pg.heatmap(data, attrs=genes, groupby='louvain_labels')

In [ ]:
data.obs.barcode = [bc for bc in data.obs.index]

In [ ]:
data.obs.barcode

In [ ]:
data.obs.barcode.to_csv("barcodes_20231104.csv")

In [ ]:
copykat = pd.read_excel("/cluster/home/yflu/STS/pegasus/copykat.pred_20231104.xlsx")

In [ ]:
copykat_1=np.array(copykat) 

In [ ]:
copykat_1[:,0]

In [ ]:
copykat_1[:,1]

In [ ]:
data.obs["copykat"] = pd.Categorical(copykat_1[:,1])

In [ ]:
pg.scatter(data,attrs=['copykat',"Tissue","Tissue_1","louvain_labels"],basis='umap',ncols=2,dpi=150)

In [ ]:
pg.scatter(data,attrs=['copykat'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.write_output(data,'/cluster/home/yflu/STS/pegasus/STS_88samples_nomiro_filter_de_copykat_beforeanno20231104.zarr')

In [ ]:
data = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88samples_nomiro_filter_de_copykat_beforeanno20231104.zarr')

In [ ]:
pg.scatter(data,attrs=['Disease'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.scatter(data,attrs=['Tissue','Tissue_1'],basis='umap',ncols=2,dpi=150)

In [ ]:
genes = ["MYLPF","MYL1","MYOD1","COL3A1","CD34", "ACTA2", "DLK1","S100B","TAGLN","COL1A1","ENG","VWF","PTPRC","EPCAM","CD3D","CD8A","CD4","NKG7","GNLY","CD14","FCGR3A","AIF1","CD68","LILRA4","CLEC4C","LYZ"]

In [ ]:
pg.heatmap(data, attrs=genes, groupby='louvain_labels')

In [ ]:
pg.violin(data, ["CHODL","EMILIN1"], groupby='louvain_labels')

In [ ]:
pg.scatter(data, ["MYOD1","MYOG"], groupby='louvain_labels')

In [ ]:
pg.scatter(data, ["PTPRC"], groupby='louvain_labels')

In [ ]:
data.obs['Celltype']=data.obs.louvain_labels.copy()
data.obs.Celltype.replace(['1'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['2'],'T cells',inplace=True)
data.obs.Celltype.replace(['3'],'T cells',inplace=True)
data.obs.Celltype.replace(['4'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['5'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['6'],'Endothelial cells',inplace=True)
data.obs.Celltype.replace(['7'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['8'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['9'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['10'],'B cells',inplace=True)
data.obs.Celltype.replace(['11'],'Pericytes',inplace=True)
data.obs.Celltype.replace(['12'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['13'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['14'],'Lipoblastoma cells',inplace=True)
data.obs.Celltype.replace(['15'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['16'],'NK cells',inplace=True)
data.obs.Celltype.replace(['17'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['18'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['19'],'MRT cells',inplace=True)
data.obs.Celltype.replace(['20'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['21'],'Undifferentiated sarcoma cells',inplace=True)
data.obs.Celltype.replace(['22'],'EWS/PNET cells',inplace=True)
data.obs.Celltype.replace(['23'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['24'],'Endothelial tumor cells',inplace=True)
data.obs.Celltype.replace(['25'],'Epithelial cells',inplace=True)
data.obs.Celltype.replace(['26'],'T cells',inplace=True)
data.obs.Celltype.replace(['27'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['28'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['29'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['30'],'EWS/PNET cells',inplace=True)
data.obs.Celltype.replace(['31'],'Fibroblasts',inplace=True)
data.obs.Celltype.replace(['32'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['33'],'MPNST cells',inplace=True)
data.obs.Celltype.replace(['34'],'MPNST cells',inplace=True)
data.obs.Celltype.replace(['35'],'MRT cells',inplace=True)
data.obs.Celltype.replace(['36'],'MRT cells',inplace=True)
data.obs.Celltype.replace(['37'],'Infantile fibrosarcoma cells',inplace=True)
data.obs.Celltype.replace(['38'],'Schwannoma cells',inplace=True)
data.obs.Celltype.replace(['39'],'Schwann cells',inplace=True)
data.obs.Celltype.replace(['40'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['41'],'RMS cells',inplace=True)
data.obs.Celltype.replace(['42'],'Epithelial cells',inplace=True)
data.obs.Celltype.replace(['43'],'Muscle cells',inplace=True)
data.obs.Celltype.replace(['44'],'EWS/PNET cells',inplace=True)
data.obs.Celltype.replace(['45'],'Epithelial cells',inplace=True)
data.obs.Celltype.replace(['46'],'MRT cells',inplace=True)
data.obs.Celltype.replace(['47'],'Mast cells',inplace=True)
data.obs.Celltype.replace(['48'],'pDC',inplace=True)
data.obs.Celltype.replace(['49'],'T cells',inplace=True)
data.obs.Celltype.replace(['50'],'Plasma cells',inplace=True)
data.obs.Celltype.replace(['51'],'Aggressive fibromatosis cells',inplace=True)
data.obs.Celltype.replace(['52'],'MRT cells',inplace=True)
data.obs.Celltype.replace(['53'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['54'],'T cells',inplace=True)
data.obs.Celltype.replace(['55'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['56'],'Juvenile xanthogranuloma endothelial cells',inplace=True)
data.obs.Celltype.replace(['57'],'Juvenile xanthogranuloma fibroblasts',inplace=True)
data.obs.Celltype.replace(['58'],'Monocytes/macrophages',inplace=True)
data.obs.Celltype.replace(['59'],'Endothelial cells',inplace=True)
data.obs.Celltype.value_counts()

In [ ]:
pg.scatter(data,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.write_output(data,'/cluster/home/yflu/STS/pegasus/STS_88samples_nomiro_filter_de_copykat_afteranno20231104.zarr')

In [ ]:
data = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88samples_nomiro_filter_de_copykat_afteranno20231104.zarr')

In [ ]:
data_tumor = data[data.obs.Celltype.isin(["Fibroblasts","Endothelial cells","RMS cells","Pericytes","Lipoblastoma cells","MRT cells","Undifferentiated sarcoma cells","EWS/PNET cells","Epithelial cells","MPNST cells","Infantile fibrosarcoma cells","Schwannoma cells","Schwann cells","Muscle cells","Aggressive fibromatosis cells","Juvenile xanthogranuloma endothelial cells","Juvenile xanthogranuloma fibroblasts","Endothelial tumor cells"])].copy()

In [ ]:
pg.scatter(data_tumor,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
data_tumor

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_88_tumor_raw_20231107.zarr.zip')

In [ ]:
data_tumor = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88_tumor_raw_20231107.zarr.zip')

In [ ]:
pg.highly_variable_features(data_tumor)

In [ ]:
pg.pca(data_tumor)

In [ ]:
pg.regress_out(data_tumor, attrs=['G1/S', 'G2/M'], rep = 'pca')

In [ ]:
pca_key = pg.run_harmony(data_tumor,rep = 'pca_regressed')

In [ ]:
pg.neighbors(data_tumor,rep=pca_key,use_cache = False)
pg.louvain(data_tumor,rep=pca_key,class_label='louvain_labels')
pg.umap(data_tumor,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_88_tumor_umap_20231107.zarr.zip')

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_88_tumor_umap_20231107.h5ad')

In [ ]:
data_tumor = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_88_tumor_umap_20231107.zarr.zip')

In [ ]:
data.obs['Disease']=data.obs.Channel_1.copy()
data.obs.Disease.replace(['CH1'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['CH3'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['IH1'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['KHE001'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE004'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE006'],'KHE',inplace=True)
data.obs.Disease.replace(['KHE009'],'KHE',inplace=True)
data.obs.Disease.replace(['T1088'],'Schwannoma',inplace=True)
data.obs.Disease.replace(['T1091'],'MPNST',inplace=True)
data.obs.Disease.replace(['T1091pi'],'MPNST - skin',inplace=True)
data.obs.Disease.replace(['T1100'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['T1100N'],'Undifferentiated sarcoma - liver',inplace=True)
data.obs.Disease.replace(['T1140'],'RMS',inplace=True)
data.obs.Disease.replace(['T1145'],'IMT',inplace=True)
data.obs.Disease.replace(['T1242'],'Schwannoma',inplace=True)
data.obs.Disease.replace(['T1250'],'RMS',inplace=True)
data.obs.Disease.replace(['T1250M'],'RMS',inplace=True)
data.obs.Disease.replace(['T1251'],'MRT',inplace=True)
data.obs.Disease.replace(['T1254'],'IMT',inplace=True)
data.obs.Disease.replace(['T1254N'],'IMT - muscle',inplace=True)
data.obs.Disease.replace(['T1311'],'RMS',inplace=True)
data.obs.Disease.replace(['T1314'],'Angiosarcoma',inplace=True)
data.obs.Disease.replace(['T1314N'],'Angiosarcoma - liver',inplace=True)
data.obs.Disease.replace(['T1335'],'MRT',inplace=True)
data.obs.Disease.replace(['T137'],'RMS',inplace=True)
data.obs.Disease.replace(['T143'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T1620'],'RMS',inplace=True)
data.obs.Disease.replace(['T1642'],'RMS',inplace=True)
data.obs.Disease.replace(['T1651'],'NF/perineurioma',inplace=True)
data.obs.Disease.replace(['T1699'],'STS',inplace=True)
data.obs.Disease.replace(['T196'],'MRT',inplace=True)
data.obs.Disease.replace(['T235'],'RMS',inplace=True)
data.obs.Disease.replace(['T347'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T353'],'RMS',inplace=True)
data.obs.Disease.replace(['T356'],'RMS',inplace=True)
data.obs.Disease.replace(['T391'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['T392'],'RMS',inplace=True)
data.obs.Disease.replace(['T442'],'MRT',inplace=True)
data.obs.Disease.replace(['T457'],'MRT',inplace=True)
data.obs.Disease.replace(['T487'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T500'],'MRT',inplace=True)
data.obs.Disease.replace(['T503'],'MRT',inplace=True)
data.obs.Disease.replace(['T508'],'IMT',inplace=True)
data.obs.Disease.replace(['T532'],'RMS',inplace=True)
data.obs.Disease.replace(['T533'],'RMS',inplace=True)
data.obs.Disease.replace(['T552'],'Synovial sarcoma',inplace=True)
data.obs.Disease.replace(['T559'],'RMS',inplace=True)
data.obs.Disease.replace(['T599'],'IMT',inplace=True)
data.obs.Disease.replace(['T601'],'RMS',inplace=True)
data.obs.Disease.replace(['T610'],'Juvenile xanthogranuloma',inplace=True)
data.obs.Disease.replace(['T611'],'Myofibroblastic tumor',inplace=True)
data.obs.Disease.replace(['T614'],'RMS',inplace=True)
data.obs.Disease.replace(['T614L'],'RMS - lymph node',inplace=True)
data.obs.Disease.replace(['T640T1'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T640T2'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T64'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T719'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T729'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T779'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T789'],'Aggressive fibromatosis',inplace=True)
data.obs.Disease.replace(['T810'],'IMT',inplace=True)
data.obs.Disease.replace(['T810N'],'IMT - lung',inplace=True)
data.obs.Disease.replace(['T822'],'RMS',inplace=True)
data.obs.Disease.replace(['T827'],'RMS',inplace=True)
data.obs.Disease.replace(['T835'],'RMS',inplace=True)
data.obs.Disease.replace(['T840'],'IMT',inplace=True)
data.obs.Disease.replace(['T854'],'MPNST',inplace=True)
data.obs.Disease.replace(['T868'],'Hemangioma',inplace=True)
data.obs.Disease.replace(['T883'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T887'],'Lipoblastoma',inplace=True)
data.obs.Disease.replace(['T888'],'Spindle cell tumor',inplace=True)
data.obs.Disease.replace(['T924T1'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T924T2'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T937'],'Pecoma',inplace=True)
data.obs.Disease.replace(['T939'],'RMS',inplace=True)
data.obs.Disease.replace(['T941'],'RMS',inplace=True)
data.obs.Disease.replace(['T943'],'EWS/PNET',inplace=True)
data.obs.Disease.replace(['T943N'],'EWS/PNET - muscle',inplace=True)
data.obs.Disease.replace(['T947C'],'NF',inplace=True)
data.obs.Disease.replace(['T963'],'RMS',inplace=True)
data.obs.Disease.replace(['T966'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T969'],'NF',inplace=True)
data.obs.Disease.replace(['T969N'],'NF - stomach',inplace=True)
data.obs.Disease.replace(['T976'],'Lymphangioma',inplace=True)
data.obs.Disease.replace(['T98'],'Infantile fibrosarcoma',inplace=True)
data.obs.Disease.replace(['T997'],'Undifferentiated sarcoma',inplace=True)
data.obs.Disease.replace(['XM6'],'EWS/PNET',inplace=True)
data.obs.Disease.value_counts()

In [ ]:
pg.scatter(data_tumor,attrs=['Disease'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.scatter(data_tumor,attrs=['Disease'],basis='tsne',ncols=1,dpi=150)

In [ ]:
pg.scatter(data_tumor,attrs=['copykat'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.scatter(data_tumor,attrs=['louvain_labels'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.scatter(data_tumor,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.de_analysis(data_tumor,cluster='louvain_labels',de_key='louvain_labels_tumor_de')

In [ ]:
marker_dict_tumor = pg.markers(data_tumor,de_key='louvain_labels_tumor_de')

In [ ]:
pg.write_results_to_excel(marker_dict_tumor, "/cluster/home/yflu/STS/pegasus/louvain_labels_tumor_de20230613.xlsx")

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_76_tumorsamples_nomiro_filter_copykat_afterDE20230613.zarr')

In [ ]:
data_tumor = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_76_tumorsamples_nomiro_filter_copykat_afterDE20230613.zarr')

In [ ]:
pg.violin(data_tumor, ["DLK1","TBX2","BBC3","ZFAS1","SCG2"], groupby='louvain_labels')

In [ ]:
pg.scatter(data_tumor, ["GDF15","TBX2","BBC3","ZFAS1","SCG2"])

In [ ]:
data_tumor.obs['CelltypeT']=data_tumor.obs.louvain_labels.copy()
data_tumor.obs.CelltypeT.replace(['1'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['2'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['3'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['4'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['5'],'Vascular endothelial cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['6'],'Proliferating cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['7'],'Muscle cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['8'],'Pericytes',inplace=True)
data_tumor.obs.CelltypeT.replace(['9'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['10'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['11'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['12'],'Immune cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['13'],'Muscle cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['14'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['15'],'Schwann cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['16'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['17'],'Lymphatic endothelial cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['18'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['19'],'Muscle cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['20'],'Hepatocytes',inplace=True)
data_tumor.obs.CelltypeT.replace(['21'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['22'],'Hepatocytes',inplace=True)
data_tumor.obs.CelltypeT.replace(['23'],'Epithelial cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['24'],'Melanocytes',inplace=True)
data_tumor.obs.CelltypeT.replace(['25'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['26'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['27'],'Mesenchymal cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['28'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['29'],'Neuroendocrine cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['30'],'Dermal fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['31'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['32'],'Epithelial cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['33'],'Epithelial cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['34'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['35'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['36'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['37'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['38'],'Smooth musecle cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['39'],'Smooth musecle cells',inplace=True)
data_tumor.obs.CelltypeT.replace(['40'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.replace(['41'],'Fibroblasts',inplace=True)
data_tumor.obs.CelltypeT.value_counts()

In [ ]:
pg.scatter(data_tumor,attrs=['CelltypeT'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.compo_plot(data_tumor, 'Disease', 'CelltypeT')

In [ ]:
pg.compo_plot(data_tumor, 'Disease', 'copykat',sort_function = "natsorted")

In [ ]:
help(pg.compo_plot)

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_76_tumorsamples_nomiro_filter_copykat_afterANNO20230619.zarr')

In [ ]:
data_tumor = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_76_tumorsamples_nomiro_filter_copykat_afterANNO20230619.zarr')

In [ ]:
pg.write_output(data_tumor,'/cluster/home/yflu/STS/pegasus/STS_76_tumorsamples_nomiro_filter_copykat_afterANNO20230619.h5ad')

In [ ]:
data = pg.read_input('/cluster/home/yflu/STS/pegasus/STS_76samples_nomiro_filter_de_copykat_afteranno20230611.zarr')

In [ ]:
pg.scatter(data,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
data_T = data[data.obs.Celltype.isin(["T cells"])].copy()

In [ ]:
pg.scatter(data_T,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.highly_variable_features(data_T)
pg.pca(data_T)
pca_key = pg.run_harmony(data_T,rep = 'pca')  
pg.regress_out(data_T, attrs=['G1/S', 'G2/M'], rep = 'pca_harmony')

In [ ]:
pg.neighbors(data_T,rep="pca_harmony_regressed")
pg.louvain(data_T,rep="pca_harmony_regressed",class_label='louvain_labels')
pg.umap(data_T,rep="pca_harmony_regressed",out_basis='umap')
pg.tsne(data_T,rep="pca_harmony_regressed",out_basis='tsne')
pg.louvain(data_T,rep="pca_harmony_regressed",resolution=0.6,class_label='louvain_labels_2')
pg.louvain(data_T,rep="pca_harmony_regressed",resolution=0.8,class_label='louvain_labels_3')
pg.louvain(data_T,rep="pca_harmony_regressed",resolution=1,class_label='louvain_labels_4')
pg.louvain(data_T,rep="pca_harmony_regressed",resolution=1.2,class_label='louvain_labels_5')

In [ ]:
pg.write_output(data_T,'/cluster/home/yflu/STS/pegasus/STS_76_T_cells_nomiro_filter_de_copykat_afterumap20230613.zarr')

In [ ]:
pg.scatter(data_T,attrs=['louvain_labels'],basis='umap',ncols=1,dpi=150)

In [ ]:
data_M = data[data.obs.Celltype.isin(["Monocyte/macrophage","Plasmacytoid dendtritic cells"])].copy()

In [ ]:
pg.scatter(data_M,attrs=['Celltype'],basis='umap',ncols=1,dpi=150)

In [ ]:
pg.highly_variable_features(data_M)
pg.pca(data_M)
pca_key = pg.run_harmony(data_M,rep = 'pca')  
pg.regress_out(data_M, attrs=['G1/S', 'G2/M'], rep = 'pca_harmony')

In [ ]:
pg.neighbors(data_M,rep="pca_harmony_regressed")
pg.louvain(data_M,rep="pca_harmony_regressed",class_label='louvain_labels')
pg.umap(data_M,rep="pca_harmony_regressed",out_basis='umap')
pg.tsne(data_M,rep="pca_harmony_regressed",out_basis='tsne')
pg.louvain(data_M,rep="pca_harmony_regressed",resolution=0.6,class_label='louvain_labels_2')
pg.louvain(data_M,rep="pca_harmony_regressed",resolution=0.8,class_label='louvain_labels_3')
pg.louvain(data_M,rep="pca_harmony_regressed",resolution=1,class_label='louvain_labels_4')
pg.louvain(data_M,rep="pca_harmony_regressed",resolution=1.2,class_label='louvain_labels_5')